In [1]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import iter_transaction_chunks

csv_path = project_root / "data/raw/archive/HI-Small_Trans.csv"
cache_dir = project_root / "data/processed/lab09"
cache_dir.mkdir(parents=True, exist_ok=True)

In [4]:
sample = pd.read_csv(csv_path, nrows=100_000)

total_rows = 5_078_345
sample_mib = sample.memory_usage(deep=True).sum() / 1024**2
estimated_mib = sample_mib * total_rows / len(sample)

print(f"Sample: {sample_mib:.1f} MiB")
print(f"Estimated full table: {estimated_mib:.1f} MiB")

Sample: 14.4 MiB
Estimated full table: 729.5 MiB


In [5]:
def load_sequence_events(path: Path) -> pd.DataFrame:
    """Load transaction history without laundering labels."""
    columns = [
        "timestamp",
        "sender_bank",
        "sender_account",
        "amount_paid",
        "payment_currency",
        "payment_format",
    ]
    string_columns = [
        "sender_bank",
        "sender_account",
        "payment_currency",
        "payment_format",
    ]

    parts = []
    row_offset = 0

    for chunk in iter_transaction_chunks(path, usecols=columns):
        # Preserve the original CSV row identity before filtering.
        chunk["source_row"] = range(
            row_offset, row_offset + len(chunk)
        )
        row_offset += len(chunk)

        chunk["timestamp"] = pd.to_datetime(
            chunk["timestamp"],
            format="%Y/%m/%d %H:%M",
            errors="raise",
        )

        in_period = (
            chunk["timestamp"].ge("2022-09-01")
            & chunk["timestamp"].lt("2022-09-09")
        )
        selected = chunk.loc[in_period].copy()

        if selected.empty:
            continue

        for column in string_columns:
            selected[column] = selected[column].astype(
                "string[pyarrow]"
            )

        parts.append(selected)

    events = (
        pd.concat(parts, ignore_index=True)
        .sort_values(["timestamp", "source_row"])
        .reset_index(drop=True)
    )
    parts.clear()

    for column in ["payment_currency", "payment_format"]:
        events[column] = events[column].astype("category")

    return events

In [6]:
events = load_sequence_events(csv_path)

assert events["source_row"].is_unique
assert events["timestamp"].is_monotonic_increasing

events.to_parquet(
    cache_dir / "sequence_events.parquet",
    index=False,
)

memory_mib = events.memory_usage(deep=True).sum() / 1024**2

print(f"History transactions: {len(events):,}")
print(f"DataFrame memory: {memory_mib:.1f} MiB")
display(events.head())

History transactions: 4,214,445
DataFrame memory: 227.7 MiB


,timestamp,sender_bank,sender_account,amount_paid,payment_currency,payment_format,source_row
0,2022-09-01,03209,8000F4670,14675.57,US Dollar,Reinvestment,2
1,2022-09-01,01420,8005DFEB0,897.37,US Dollar,Reinvestment,17
2,2022-09-01,010,8000F6850,99986.94,US Dollar,Reinvestment,158
3,2022-09-01,012,8001167D0,16.08,US Dollar,Reinvestment,218
4,2022-09-01,001,8001364A0,10.30,US Dollar,Reinvestment,281


In [7]:
in_training = (
    events["timestamp"].ge("2022-09-03")
    & events["timestamp"].lt("2022-09-07")
)

account_counts = (
    events.loc[in_training]
    .groupby(["sender_bank", "sender_account"])
    .size()
)
account_counts

sender_bank  sender_account
001          800042CB0           36
             800054E60           76
             8000555D0           38
             800056160           25
             800056370           44
                               ... 
070          100428930         1994
             100428978         6316
             1004289C0         5156
             100428A08         2564
             100428A51         4030
Length: 145740, dtype: int64

In [13]:
in_training = (
    events["timestamp"].ge("2022-09-03")
    & events["timestamp"].lt("2022-09-07")
)

account_counts = (
    events.loc[in_training]
    .groupby(["sender_bank", "sender_account"])
    .size()
)

example_bank, example_account = account_counts.idxmax()

same_account = (
    events["sender_bank"].eq(example_bank)
    & events["sender_account"].eq(example_account)
)
account_events = events.loc[same_account].copy()

account_training = account_events.loc[
    account_events["timestamp"].ge("2022-09-03")
    & account_events["timestamp"].lt("2022-09-07")
]

# The transaction whose available history we want to inspect.
target = account_training.iloc[-1]
target_time = target["timestamp"]

available_history = account_events.loc[
    account_events["timestamp"].ge(
        target_time - pd.Timedelta(hours=24)
    )
    & account_events["timestamp"].lt(target_time)
]

prior_events = available_history.tail(8).copy()
prior_events["minutes_before_target"] = (
    target_time - prior_events["timestamp"]
).dt.total_seconds() / 60

display_columns = [
    "source_row",
    "timestamp",
    "amount_paid",
    "payment_currency",
    "payment_format",
]

print("Transaction to score:")
display(target[display_columns].to_frame().T)

print("Earlier transactions available to the model:")
display(
    prior_events[
        display_columns + ["minutes_before_target"]
    ]
)

Transaction to score:


,source_row,timestamp,amount_paid,payment_currency,payment_format
3248708,3242895,2022-09-06 23:59:00,73267.16,US Dollar,Cash


Earlier transactions available to the model:


,source_row,timestamp,amount_paid,payment_currency,payment_format,minutes_before_target
3248290,3240822,2022-09-06 23:58:00,897705.29,US Dollar,Cash,1.0
3248304,3241293,2022-09-06 23:58:00,58.74,US Dollar,Cash,1.0
3248305,3241317,2022-09-06 23:58:00,3.10,US Dollar,Cheque,1.0
3248308,3241404,2022-09-06 23:58:00,309.02,US Dollar,Cheque,1.0
3248320,3241708,2022-09-06 23:58:00,180.84,US Dollar,Cheque,1.0
3248351,3242530,2022-09-06 23:58:00,197.02,US Dollar,Cheque,1.0
3248354,3242611,2022-09-06 23:58:00,579.33,US Dollar,Cheque,1.0
3248387,3243756,2022-09-06 23:58:00,619.69,US Dollar,Cheque,1.0


In [ ]:
# test = account_training.iloc[-3:,:]
# testAllow notices from a Cinder account.

,timestamp,sender_bank,sender_account,amount_paid,payment_currency,payment_format,source_row
3248700,2022-09-06 23:59:00,070,100428660,125.20,US Dollar,Credit Card,3242553
3248703,2022-09-06 23:59:00,070,100428660,472.57,US Dollar,Credit Card,3242602
3248708,2022-09-06 23:59:00,070,100428660,73267.16,US Dollar,Cash,3242895


In [24]:
assert 0 < len(prior_events) <= 8

assert prior_events["timestamp"].lt(target_time).all()
assert prior_events["timestamp"].ge(
    target_time - pd.Timedelta(hours=24)
).all()

assert prior_events["sender_bank"].eq(example_bank).all()
assert prior_events["sender_account"].eq(example_account).all()

assert target["source_row"] not in prior_events["source_row"].values
assert prior_events["timestamp"].is_monotonic_increasing

print("History checks passed.")

History checks passed.
